## 🇧🇷 Brazilian Grand Prix | 2025

In this notebook we focus on debriefing the Brazilian GP, 2025. I am currently ironing out my high-level API and ingridients that compose the recipe of a race debrief and will continue to evolve over the next few notebooks.

### Debrief Recipe (4 Pillars)
- Car Setup and Circuit Characteristics corelation.
- Car Dynamics and Management Delta.
- Tyre Degradation Coefficients.
- Attention Flags for best Car and Driving Style (wrt Telemetry; if available).

### Notebook Configurations

In [ ]:
from pathlib import Path
import sys


# Accessing the Current Working Directory for Absolute Root Path
REPO_ROOT_PATH = Path.cwd().parent.parent.parent

# Pathlib Path Objects for the Notebook
F1_PATH = REPO_ROOT_PATH / "F1"
CACHE_PATH = F1_PATH / "cache"
CACHE_PATH.mkdir(exist_ok=True)

# Adding the source scripts into the Python Execution Path
if F1_PATH not in sys.path:
    sys.path.append(str(F1_PATH))

### Imports

In [ ]:
# FastF1 Dependencies
import fastf1
from fastf1.core import Laps
from fastf1.core import CircuitInfo
from fastf1.core import Lap

# Visualisation Dependencies
from plotly.offline import init_notebook_mode

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Codebase Dependencies
from src.custom import CustomSession
from src.data import DataUtils
from src.pipeline import Pipeline
from src.plotting import Plotting

# Enabling Notebook Rendering for Plotly
init_notebook_mode(connected=True)

### Accessing the Specific Event and Sessions

In [ ]:
# Defaulting the Requests for Caching to a Predefined path
# fastf1.Cache.clear_cache(cache_dir=str(CACHE_PATH))
fastf1.Cache.enable_cache(cache_dir=str(CACHE_PATH))

# Enabling Offline Mode
if CACHE_PATH.exists():
    fastf1.Cache.offline_mode(enabled=True)
else:
    fastf1.Cache.offline_mode(enabled=False)

# The GP Event
brazil_2025 = fastf1.get_event(
    year=2025,
    gp="brazil"
)
print(brazil_2025)

### Source Dependency Instances

In [ ]:
# Data Handler: Loading and Storing Data from FastF1
data_handle = DataUtils(race_event=brazil_2025, cache_dir=str(CACHE_PATH))

# Data Pipeline: Transforming Data
data_pipeline = Pipeline()

# Plot Handler: Plotting Categorical, Joint, Marginal Distributions and Telemetry Traces
plot_handle = Plotting()

### Loading the Data for the Event

In [ ]:
brazil_quali, brazil_race = data_handle.load_data()
print(brazil_quali)

### CustomSession objects for each FastF1 Session

In [ ]:
# Accessing the different data streams for each session
qualifying_session = CustomSession(session=brazil_quali)
race_session = CustomSession(session=brazil_race)

### Data Cleaning and Segregation

**Accessing the top 5 drivers for each of the sessions**

In [ ]:
top_5_drivers_in_race = race_session.results.iloc[:5]["DriverNumber"]
print("Top 5 Drivers in the Race:")
print(top_5_drivers_in_race)

top_5_drivers_in_quali = qualifying_session.results.iloc[:5]["DriverNumber"]
print("\nTop 5 Drivers in the Qualifying:")
print(top_5_drivers_in_quali)

**Picking only the quick laps from the sessions for the top drivers (in the race)**<br><br>

⚠️ *As much as Will Buxton and DTS have gotten to me the reasoning is straight-forward points arent awarded for qualifying and hence all the analysis is done based on the results of the race*

✅ `However a separate Quali Lap Telemetry based dominance will also be provided in the analysis`

In [ ]:
race_fast_laps = (
    race_session.laps
    .pick_drivers(identifiers=top_5_drivers_in_race)  
    .pick_quicklaps()
)

quali_fast_laps_by_quali = (
    qualifying_session.laps
    .pick_drivers(identifiers=top_5_drivers_in_quali)  
    .pick_quicklaps()
)

q1_laps, _, q3_laps = qualifying_session.laps.split_qualifying_sessions()
assert isinstance(q1_laps, Laps)

quali_fast_laps_by_race = (
    q1_laps
    .pick_drivers(identifiers=top_5_drivers_in_race)
    .pick_quicklaps()
)
quali_fast_laps_by_race.head()

**Segregating the Telemetry among Mini-Sessions**<br><br>
⚠️ *Telemetry only available for Qualifying*

In [ ]:
assert isinstance(q1_laps, Laps)
assert isinstance(q3_laps, Laps)

# Lookup tables for the quick-lap driven telemetry
q1_telemetry_digest, q3_telemetry_digest = {}, {}

# Iterating through the top 5 drivers to access telemetetry from Qualifying
for driver_number in top_5_drivers_in_quali:
    q1_telemetry = (
        q1_laps.pick_drivers(identifiers=driver_number)
        .pick_fastest()
        .get_car_data()  # type: ignore
        .add_differential_distance()  # Delta Distance covered between two samples
        .add_distance()  # Cumulative Distance covered from first sample
    )
    q1_telemetry_digest[driver_number] = q1_telemetry

    q3_telemetry = (
        q3_laps.pick_drivers(identifiers=driver_number)
        .pick_fastest()  
        .get_car_data()  # type: ignore
        .add_differential_distance()
        .add_distance()
    )
    q3_telemetry_digest[driver_number] = q3_telemetry

q1_telemetry_digest.keys()

In [ ]:
# Race based telemetry digest
q1_telemetry_by_race_digest = {}

for driver_number in top_5_drivers_in_race:
    q1_telemetry_by_race_result = (
        quali_fast_laps_by_race.pick_drivers(identifiers=driver_number)
        .pick_fastest()
        .get_car_data()  # type: ignore
        .add_differential_distance()
        .add_distance()
    )
    q1_telemetry_by_race_digest[driver_number] = q1_telemetry_by_race_result

q1_telemetry_by_race_digest.keys()

**Windowing Minisectors on the Track by Distance**

In [ ]:
# Iterating through the driver telemetry for each session to apply windowing
for driver_number in top_5_drivers_in_quali:
    
    # Making a copy to avoid pd.futurewarning
    # Mapping the telemetry keypoints for Q1
    copy_tel_q1 = q1_telemetry_digest[driver_number].copy()
    copy_tel_q1 = data_pipeline.map_telemetry_keypoints(copy_frame=copy_tel_q1)
    q1_telemetry_digest[driver_number] = copy_tel_q1

    # Mapping the telemetry keypoints for Q3
    copy_tel_q3 = q3_telemetry_digest[driver_number].copy()
    copy_tel_q3 = data_pipeline.map_telemetry_keypoints(copy_frame=copy_tel_q3)
    q3_telemetry_digest[driver_number] = copy_tel_q3

# Testing the output transformation
q3_telemetry_digest["81"].head()

In [ ]:
# Iterating through the driver telemetry for q1 based on the race result
for driver_number in top_5_drivers_in_race:
    
    copy_tel_q1_by_race = q1_telemetry_by_race_digest[driver_number].copy()
    copy_tel_q1_by_race = data_pipeline.map_telemetry_keypoints(copy_frame=copy_tel_q1_by_race)
    q1_telemetry_by_race_digest[driver_number] = copy_tel_q1_by_race

# Testing the output transformation
q1_telemetry_by_race_digest["1"].head()

**Visualising a sample telemetry trace**

In [ ]:
plot_handle.plot_driver_telemetry_traces(
    top_5_driver_telemetry=q3_telemetry_digest, show_distance=True
)

**Charting the Circuit Map**

In [ ]:
# Accessing the Circuit Information from the Session Object
circuit_info = brazil_race.get_circuit_info()
assert isinstance(circuit_info, CircuitInfo)

circuit_layout = circuit_info.corners
circuit_rotation = circuit_info.rotation

circuit_angle_radians =  circuit_rotation * (np.pi/180)
print(f"Circuit Layout by Corners:\n{circuit_layout}")
print(f"\nCircuit Rotation: {circuit_rotation}")

In [ ]:
def rotate_points(xy_vector: np.ndarray, *, angle: float) -> np.ndarray:
    # Rotation Matrix for a 2D cartesian system
    rotation_matrix = np.array([
        [np.cos(angle), np.sin(angle)],
        [-np.sin(angle), np.cos(angle)]
    ])

    # Rotation of the Vector wrt Circuit Specifications
    return xy_vector @ rotation_matrix

# Getting the position data for rotation using the Circuit Spec
assert isinstance(q3_laps, Laps)
fastest_lap = q3_laps.pick_fastest()

assert isinstance(fastest_lap, Lap)
pos_data = fastest_lap.get_pos_data()

track_data = pos_data.loc[:, ["X", "Y"]].to_numpy()
rotated_track = rotate_points(xy_vector=track_data, angle=circuit_angle_radians)

plt.plot(rotated_track[:, 0], rotated_track[:, 1])
plt.show()

In [ ]:
pos_data

### Feature Engineering

**Qualifying Lap Throttle Percentage**<br><br>
⚠️ *Based on race results to calculate throttle usage per lap*

In [ ]:
# Throttle Map Lookup Table
throttle_map_digest = {}

# Iterating through the drivers to calculate the driver throttle maps
for driver_number in top_5_drivers_in_race:
    throttle_map = data_handle.get_throttle_map(
        driver_quali_telemetry=q1_telemetry_by_race_digest[driver_number]
    )
    throttle_map_digest[driver_number] = throttle_map

throttle_map_digest

**Fuel Aware Laptime Correction**

In [ ]:
# Initialising the new columns
race_fast_laps["LapFuelBurn"] = 0.0
race_fast_laps["CumulativeFuelBurn"] = 0.0
race_fast_laps["FuelAwareLapTime"] = 0.0

# Iterating through the drivers to correct the laptimes
for driver_number in top_5_drivers_in_race:
    
    # Accessing the Drivers Laps and Throttle Map
    driver_laps = race_fast_laps.pick_drivers(identifiers=driver_number).copy()
    driver_throttle_map = throttle_map_digest[driver_number]

    # Tranforming the frame for the Fuel Aware Laptimes
    corrected_laptimes = data_pipeline.get_fuel_aware_laptime(
        driver_laps=driver_laps,
        driver_throttle_handle=driver_throttle_map
    )

    # Moving the changes to the orignal laps frame
    race_fast_laps[race_fast_laps["DriverNumber"] == driver_number] = corrected_laptimes

race_fast_laps.groupby("Driver")["CumulativeFuelBurn"].last()

**Efficiency Index: Top Speed Vs Avg Speed (Cornering Load)**

In [ ]:
eff_digest_cs = {}

# Iterating through each driver to calculate the efficiency index
for driver_number in top_5_drivers_in_race:
    driver_telemetry = q1_telemetry_by_race_digest[driver_number]

    # Calculating the Driver Specific map of estimates for each keypoint
    driver_q1_fastest_trace = (
        driver_telemetry
        .groupby("Keypoint", observed=False)["Speed"]
        .agg(["mean", "max"])
    )

    # Calculating the Efficiency Index
    eff_index_cs = data_pipeline.get_efficiency_index_corner_to_straight(
        driver_point_estimates_by_trace=driver_q1_fastest_trace
    )

    eff_digest_cs[driver_number] = eff_index_cs

eff_digest_cs

**Traction Energy: Throttle Vs Speed (over a full lap)**<br><br>
*Synonymous to the phrase `'putting the power on the road'`*

In [ ]:
driver_te_digest = {}
driver_lap_wise_te_digest = {}

# Iterating through all the drivers to calculate the Traction Energy
for driver_number in top_5_drivers_in_race:

    # Traction Energy by Keypoint
    te_keypoint_digest = (
        data_pipeline
        .get_keypoint_traction_energy(
            driver_telemetry_trace=q1_telemetry_by_race_digest[driver_number]
        )
    )
    driver_te_digest[driver_number] = te_keypoint_digest

    # Traction Energy over the Lap
    driver_lap_wise_te_digest[driver_number] = (
        data_pipeline.get_lap_traction_energy(driver_keypoint_map=te_keypoint_digest)
    )

    print(f"{driver_number} - {driver_lap_wise_te_digest[driver_number]}")

In [ ]:
for keypoint, (te, _) in driver_te_digest["1"].items():
    print(f"{keypoint} - {te}")

**Braking Energy - Longitudinal Stress**<br><br>
*Deceleration observed in the telemetry of each driver in a straight line*

In [ ]:
driver_be_digest = {}
driver_lap_wise_be_digest = {}

# Iterating through the top 5 drivers to find the braking energy utilisation
for driver_number in top_5_drivers_in_race:
    driver_braking_telemetry = q1_telemetry_by_race_digest[driver_number]
    
    # Keypoint wise braking energy data
    braking_keypoint_digest = (
        data_pipeline
        .get_keypoint_braking_energy(driver_telemetry=driver_braking_telemetry)
    )
    driver_be_digest[driver_number] = braking_keypoint_digest

    # Lap wise braking energy data
    driver_lap_wise_be_digest[driver_number] = (
        data_pipeline
        .get_lap_braking_energy(braking_keypoint_digest)
    )

    print(f"{driver_number} - {driver_lap_wise_be_digest[driver_number]}")

In [ ]:
for keypoint, be in driver_be_digest["1"].items():
    print(f"{keypoint} - {be}")

**Braking Force - Contact Patch wear based on Braking**<br><br>
*The constant baseline force being exerted on the contact patch provided by all the four tyres (longitudinal load)*

>**Important:**<br>
>All the forces are calculated on a mean distribution hence the peaks force isnt representative of the full picture but provides an average estimate for wear that persists over a race stint.

In [ ]:
driver_bf_digest = {}
driver_lap_wise_bf_digest = {}

# Iterating through the top 5 drivers to find the braking force
for driver_number in top_5_drivers_in_race:

    # Meta Param for Mass based on Driver
    mean_fuel_burn = race_fast_laps.pick_drivers(driver_number)["LapFuelBurn"].mean()

    # Keypoint wise braking force data
    driver_braking_energy = driver_be_digest[driver_number]
    braking_force_digest = (
        data_pipeline
        .get_keypoint_braking_force(
            driver_braking_energy_map=driver_braking_energy,
            mean_fuel_burn=mean_fuel_burn
        )
    )
    driver_bf_digest[driver_number] = braking_force_digest

    # Lap wise braking force data
    driver_lap_wise_bf_digest[driver_number] = (
        data_pipeline
        .get_lap_braking_force(braking_force_digest)
    )

for driver_number, bf in driver_lap_wise_bf_digest.items():
    print(f"{driver_number} - {bf}")

In [ ]:
for keypoint, bf in driver_bf_digest["1"].items():
    print(f"{keypoint} - {bf}")

### Multivariate Analysis

Based on the above feature engineering each driver is attributed by a 4 feature vector that best describes their performance during a race.

**Main Features**
- Efficiency Index: Cornering Vs Straight Line Load.
- Traction Energy: Correlation between Speed, Throttle and Distance covered during instantaneous samples.
- Braking Force: The force exerted on the contact patch during a lap.
- Throttle Map: The amount of time spent of full, partial and closed throttle during a single flying lap.

*Auxilary Features*
- Keypoint Normalisation: Braking down the telemetry trace into uniform windows.
- Braking Energy: Correlation between the Decelation and Distance covered during instantaneous samples (helper for Braking Force).

**Constructing the Driver Fingerprint Dataframe using Qualifying**

In [ ]:
# Ensuring that Q1 Laps is not None
assert isinstance(q1_laps, Laps)

fingerprint_df = data_handle.get_fingerprint_frame(
    q1_laps=q1_laps,
    top_5_drivers=top_5_drivers_in_race.tolist(),
    throttle_map_digest=throttle_map_digest,
    keypoint_te_digest=driver_te_digest,
    keypoint_bf_digest=driver_bf_digest,
    efficiency_digest=eff_digest_cs
)

**Fingerprint Correlations by Quali**

*Important*
- `Negative Correlations`: Contribute directly to the Pace of the Driver.
- `Positive Correlations`: Contribute directly to the Degradation of the Driver.

This is contrary to the general understanding of correlation matrices since the target feature **Laptime** is a cost over the race distance as more laptime per-lap means slower car and viz.

Therefore the a high-level observation of the correlations showcases the most corner exits and acceleration keypoints have a strong negative correlation as the smaller they are the lesser the laptime. On the contrary the corner entry / braking keypoints are strong positive correlations as they reduce the pace and cause degradation (tangential affect) which bleed laptime.

In [ ]:
plt.figure(figsize=(30, 30))
sns.heatmap(fingerprint_df.corr(), annot=True, fmt=".3f")
plt.title("Driver Fingerprint Correlations")
plt.show()

**Mapping the Engineered Features to the Race Laps Frame**

In [ ]:
# Dropping the Quali Bench Time from the Fingerprint before mapping
fingerprint_df = fingerprint_df.drop("fastest_quali_laptime", axis=1)

multivariate_df = data_handle.get_multivariate_frame(
    fingerprint_frame=fingerprint_df,
    race_fast_laps=race_fast_laps
)

multivariate_df.head()

## Fin 🏎️ ✨